# Training a classifier using PyTorch on the OSPool

## Introduction

In this tutorial, we will submit jobs using PyTorch to the OSPool to train a cat/dog classifier to distinguish between images of cats and dogs.

## Our code

Let's take a look at our wrapper script, `train.sh`.

In [ ]:
cat train.sh

<br>

**This script:**
1. Unzips our training data into a directory called `data`.
2. Runs our Python script (and saves our model checkpoint in the current working directory).

<p style="text-align: center;"><img src="img/train.png" width="600px"></p>

> 💡 **Tip: Use arguments**
>
> We have written our Python script to take in arguments. This makes it easier to modify in later training jobs, in case we need to change the number of training epochs or use different input/output directories.

## The HTCondor submit file

The HTCondor submit file, `train.sub`, describes our training task, its inputs/outputs, and the resources we need for to run the task.

The combination of these components make up a unit of work, which we call a **job**. The submit file translates our job into a format that HTCondor understands.

Let's take a look at the contents of `train.sub` for this task:

In [ ]:
cat train.sub

<br>

### A breakdown of the options

| Submit file option | Purpose |
| --- | --- |
| `batch_name` | (Optional) Gives HTCondor a name for our job, and can allow us to cluster different submissions into single batches. Default value is `$(Cluster)` |
| `container_image` | Path to the container image we use that contains our software environment. In this case, we're using a pre-built container with conda and PyTorch, accessible via a public namespace in the Open Science Data Federation. |
| `shell` | The command to run. |
| `transfer_input_files` | All input files needed for the job that need to be transferred to the machine that executes our job (Execution Point). This includes the shell script, the Python script, and training data. | 
| `transfer_output_files` | (Optional) Which output files to transfer back to the Access Point when the job is complete. If not specified, the default value is any new or changed *files* in the *top-level directory* of the Execution Point. Subdirectories and their contents are not transferred by default.
| `output` | Where to write any standard output printed to the terminal during the course of the job. |
| `error` | Where to write any standard error printed to the terminal during the course of the job. |
| `log` | Where to write HTCondor's job log file. This is a file that tracks metadata information about your job, and it is updated as your job progresses. | 
| `request_*` | Resource requests. While not strictly required, the default values are relatively small, so it's good practice to always specify these values. |
| `gpus_minimum_*` | (Optional) Requirements for narrowing the types of GPUs your jobs can run on. | 
| `should_transfer_files` | Only necessary if submitting via an OSPool notebook. |
| `queue` | Indicates the end of the submit file and "queues" one job. Can be edited to submit many similar jobs. |

## Submit a training job

To submit our training job, we use the `condor_submit` command:

In [ ]:
condor_submit train.sub

<br>
If all goes correctly, you should see a message, like:

```
Submitting job(s).
1 job(s) submitted to cluster 14915869.
```

Though your cluster/job ID will be different and unique per submission.

## Monitor your job

You can then check your job's status with `condor_q`:

In [ ]:
condor_q

Or show a real-time view using the `condor_watch_q` command. This utility tracks your log file and prints a color-coded summary of your jobs every second.

> ⚠️ **Run `condor_watch_q` in the terminal**
>
> `condor_watch_q` doesn't render very well in the Jupyter notebook, so open a new Terminal tab and run it there.

Curious to know where your job is running? We can use the `condor_q` command again with an additional flag:

In [ ]:
condor_q -af RemoteHost

<br>

*The job itself will take approximately 30 minutes to finish 10 epochs of training.*

## Check the outputs of your job

When your job is complete, it is good practice to check your outputs to see that you get the expected results. Let's list information about the log, standard output, and standard error files.

In [ ]:
ls -lh logs/

<br>

If the standard error files (`.err`) have 0 bytes in them, then you *likely* don't have any errors with your job. We can also look at the standard output files (`.out`) to check what messages were printed by our job. For brevity, we'll just `tail` the end of the output files.

In [ ]:
tail logs/*.out

<br>

Lastly, we should have `model.pth` file, which is the output file we want!

In [ ]:
ls -lh model.pth

🌟 **Congratulations!** You've trained a cat/dog classifier using the GPUs available on the Open Science Pool!

## Troubleshooting

### "My job is held. What do I do?"

When HTCondor detects an issue with your job and can't proceed any further, it puts your job into a *hold* or *held* state, marked by an `H` when running `condor_q` or red `!` symbols in `condor_watch_q`.

When this happens, we recommend starting the troubleshooting process with `condor_q -hold` to get more information:

For example:

```
[user.name@ap40]$ condor_q -hold
14915682.423   user.name  7/23 18:11   47/0   The job exceeded allowed execute duration of 20:00:00
```

The message will usually hint at what you may need to do next, whether it's to fix a typo or adjust your resource requests. If you're ever unsure what to do with a specific hold message, you can always talk to an OSG Facilitator at [support@osg-htc.org](mailto:support@osg-htc.org).

### "My job ran successfully, but I don't see expected outputs. What went wrong?"

In this case, HTCondor detected that your job ran "successfully", meaning there were no fatal errors, but your scripts/code itself may not have ran successfully. When this happens, the best place to start is to look at the standard error and output files that HTCondor sends back with your jobs. These files will contain critical information in debugging your job.